# JP 波及タイミング推定

EN ピーク日（DTW 推定）をもとに、各キーワードがいつ頃 JP でピークを迎えるかを推定する。

**推定ロジック**  
EN 上昇波形を確定 LAG_PAIRS と DTW でマッチングし、最近傍ペアのラグ（11〜16ヶ月）を EN ピーク日に加算して JP 推定ピーク日を算出する。

**分析フロー**
1. EN ピーク日の確認（DTW 推定結果）
2. JP 波及タイミング推定（dtw_analog）
3. EN / JP 系列の可視化と結果サマリー

**JP キーワード選定基準**  
各コンセプトに対応する JP キーワードを以下のルールで選定する。

| 優先順位 | ルール |
|---|---|
| 1 | 手動マッピング（`JP_MANUAL`）が定義されているものを優先（データが十分な場合のみ適用） |
| 2 | 自動選択: `mean_last52`（直近1年平均）が最大のキーワード |

手動マッピング対象（同コンセプト内で EN キーワードが異なる JP 概念を指す場合のみ）:
- `lace top outfit` → `レーストップス`（`lace outfit` コンセプトの自動選択が `レーススカート` になるため）

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')
import logging
for _name in ['cmdstanpy', 'prophet', 'stan', 'pystan']:
    logging.getLogger(_name).setLevel(logging.ERROR)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trend_utils import load_category, PERIODS, get_en_series, get_jp_series, get_mean_last52, get_status
from screen_pairs import MIN_PEAK, MIN_LAST52, get_monitoring_keywords, get_best_jp_keyword
from lag_estimator import LAG_PAIRS, estimate_en_peak_dtw, dtw_analog_match, add_months

# ── 定数 ──────────────────────────────────────────────────────────────────────
RISE_WINDOW_WEEKS = 78

# ── データ終端・今日 ──────────────────────────────────────────────────────────
DATA_END = load_category('styles_en', PERIODS[0]).index.max()
TODAY    = pd.Timestamp.now().normalize()

# ── 予測対象キーワードの構築（en_peak_prediction と同一条件）────────────────
EXCLUDED_KEYWORDS = {
    'crossbody bag', 'messenger bag',
    'military jacket', 'army jacket',
    'sequin outfit', 'mid rise jeans',
    'mob wife aesthetic', 'old money aesthetic',
}
result = get_monitoring_keywords(EXCLUDED_KEYWORDS)
result['状態']      = result['peak_date'].apply(lambda x: get_status(x, DATA_END, TODAY))
result['jp_keyword'] = result['concept'].apply(get_best_jp_keyword)

# 同コンセプト内で EN キーワードが異なる JP 概念を指す場合のみ手動上書き
# データが十分ある場合（mean_last52 >= MIN_LAST52）のみ適用
JP_MANUAL = {'lace top outfit': 'レーストップス'}
for en_kw, jp_kw in JP_MANUAL.items():
    if get_mean_last52(jp_kw, geo='jp') >= MIN_LAST52:
        result.loc[result['keyword'] == en_kw, 'jp_keyword'] = jp_kw

print(f'DATA_END : {DATA_END.strftime("%Y-%m-%d")}')
print(f'TODAY    : {TODAY.strftime("%Y-%m-%d")}')
print(f'予測対象: {len(result)}件')
result[['concept', 'keyword', 'mean_last52', '状態', 'jp_keyword']].reset_index(drop=True)

## Step 1: EN ピーク日の確認（DTW 推定）

`en_peak_prediction.ipynb` と同一条件で DTW スライディングウィンドウを実行し、  
EN ピーク日を取得する。

In [ ]:
from lag_estimator import ADOPTED_TEMPLATES, ALL_TEMPLATES

en_peaks = {}
for _, row in result.iterrows():
    kw        = row['keyword']
    en_series = get_en_series(kw)
    if en_series is None:
        continue
    obs = en_series[en_series.index >= DATA_END - pd.Timedelta(weeks=RISE_WINDOW_WEEKS - 1)]
    res = estimate_en_peak_dtw(obs, DATA_END, ALL_TEMPLATES)
    if res is None:
        continue
    en_peaks[kw] = {
        'en_series':     en_series,
        'en_peak_date':  res['en_peak_est'],
        'best_template': res['best_template'],
        'dtw_distance':  round(res['dtw_distance'], 3),
    }

In [3]:
kw_to_concept = result.set_index('keyword')['concept'].to_dict()

n = len(en_peaks)
cols = 3
rows = (n + cols - 1) // cols
fig_dtw = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f"{kw_to_concept.get(kw, '')} ({kw})" for kw in en_peaks],
    vertical_spacing=0.14, horizontal_spacing=0.07,
)

for idx, (kw, r) in enumerate(en_peaks.items()):
    ri, ci = idx // cols + 1, idx % cols + 1
    s = r['en_series']

    fig_dtw.add_trace(go.Scatter(
        x=s.index, y=s.values,
        line=dict(color='#4472C4', width=1.5), showlegend=False,
    ), row=ri, col=ci)

    fig_dtw.add_vline(x=DATA_END.timestamp() * 1000,
        line=dict(color='gray', dash='dash', width=1), row=ri, col=ci)

    fig_dtw.add_vline(x=r['en_peak_date'].timestamp() * 1000,
        line=dict(color='#2CA02C', dash='dot', width=2), row=ri, col=ci)

    fig_dtw.add_annotation(
        x=r['en_peak_date'], y=s.max(),
        text=f"<b>{r['en_peak_date'].strftime('%Y-%m')}</b>",
        showarrow=True, arrowhead=2, font=dict(size=9, color='#2CA02C'),
        row=ri, col=ci,
    )

fig_dtw.update_layout(
    title=dict(text='DTW ピーク推定（緑点線: 推定ピーク / 灰破線: DATA_END）', font=dict(size=13)),
    height=300 * rows, template='plotly_white',
)
fig_dtw.update_yaxes(range=[0, 110])
fig_dtw.show()

## Step 2: JP 波及タイミング推定

EN ピーク日をもとに、確定 LAG_PAIRS との DTW マッチングで JP 波及タイミングを推定する。

**推定ロジック（dtw_analog）**  
EN 上昇波形を確定 LAG_PAIRS（Y2K fashion / balletcore / low rise jeans / normcore）と DTW でマッチングし、  
最近傍ペアのラグ（11〜16ヶ月）を EN ピーク日に加算して JP 推定ピーク日を算出する。

> Prophet による JP 系列外挿も比較検討したが、JP データが上昇途中のキーワードに対しては  
> ラグがほぼ 0〜マイナスになり機能しないことを確認。dtw_analog を採用。

In [ ]:
jp_timing_results = []
for _, row in result.iterrows():
    kw = row['keyword']
    if kw not in en_peaks:
        continue

    en_series    = en_peaks[kw]['en_series']
    en_peak_date = en_peaks[kw]['en_peak_date']
    jp_kw        = row.get('jp_keyword')

    match  = dtw_analog_match(en_series, en_peak_date)
    jp_est = add_months(en_peak_date, match['lag_months'])

    jp_timing_results.append({
        'concept':       row['concept'],
        'EN keyword':    kw,
        'JP keyword':    jp_kw,
        'EN peak (DTW)': en_peak_date.strftime('%Y-%m'),
        'JP peak (est)': jp_est.strftime('%Y-%m'),
        'lag (months)':  match['lag_months'],
        'nearest_pair':  match['best_template'],
        'dtw_distance':  match['dtw_distance'],
        '_en_peak_ts':   en_peak_date,
        '_jp_est_ts':    jp_est,
    })

## Step 3: EN / JP 系列の可視化

EN（青）と JP（紫点線）の実績系列に、各推定値を縦線で重ねて表示する。

- 緑点線: EN ピーク推定（DTW）
- 赤点線: JP 波及推定（DTW analog）
- 灰破線: DATA_END

In [5]:
n = len(jp_timing_results)
cols = 3
rows = (n + cols - 1) // cols
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f"{r['concept']} ({r['EN keyword']})" for r in jp_timing_results],
    vertical_spacing=0.16, horizontal_spacing=0.07,
)

for idx, r in enumerate(jp_timing_results):
    ri, ci = idx // cols + 1, idx % cols + 1
    kw        = r['EN keyword']
    en_series = en_peaks[kw]['en_series']
    jp_series = get_jp_series(r['JP keyword'])

    # EN 実績
    fig.add_trace(go.Scatter(
        x=en_series.index, y=en_series.values,
        name='EN', line=dict(color='#4472C4', width=1.5),
        legendgroup='en', showlegend=(idx == 0),
    ), row=ri, col=ci)

    # JP 実績
    if jp_series is not None:
        fig.add_trace(go.Scatter(
            x=jp_series.index, y=jp_series.values,
            name=f"JP: {r['JP keyword']}", line=dict(color='#9467BD', width=1.5, dash='dot'),
            legendgroup='jp', showlegend=(idx == 0),
        ), row=ri, col=ci)

    # EN ピーク縦線（緑）
    fig.add_vline(x=r['_en_peak_ts'].timestamp() * 1000,
        line=dict(color='#2CA02C', dash='dot', width=2), row=ri, col=ci)

    # JP 推定ピーク縦線（赤）
    fig.add_vline(x=r['_jp_est_ts'].timestamp() * 1000,
        line=dict(color='#D62728', dash='dot', width=2), row=ri, col=ci)

    # DATA_END（灰）
    fig.add_vline(x=DATA_END.timestamp() * 1000,
        line=dict(color='gray', dash='dash', width=1), row=ri, col=ci)

    # EN peak アノテーション
    fig.add_annotation(
        x=r['_en_peak_ts'], y=78,
        text=f"<b>EN peak</b><br>{r['EN peak (DTW)']}",
        showarrow=False,
        font=dict(size=8, color='#2CA02C'), align='center',
        bgcolor='rgba(255,255,255,0.85)',
        row=ri, col=ci,
    )

    # JP peak アノテーション（nearest_pair含む）
    jp_kw_label   = r['JP keyword'] if r['JP keyword'] and not pd.isna(r['JP keyword']) else 'N/A'
    nearest_label = r['nearest_pair'].split(' → ')[0]  # EN部分のみ
    fig.add_annotation(
        x=r['_jp_est_ts'], y=95,
        text=(
            f"<b>JP peak</b><br>{r['JP peak (est)']}<br>"
            f"+{r['lag (months)']}M<br>"
            f"<i>analog: {nearest_label}</i><br>"
            f"<i>{jp_kw_label}</i>"
        ),
        showarrow=False,
        font=dict(size=8, color='#D62728'), align='center',
        bgcolor='rgba(255,255,255,0.85)',
        row=ri, col=ci,
    )

fig.update_layout(
    title=dict(
        text='EN/JP 系列（緑: EN peak 推定 / 赤: JP 波及推定 / 灰: DATA_END）',
        font=dict(size=13),
    ),
    height=340 * rows, template='plotly_white',
    legend=dict(orientation='h', y=-0.05, x=0.5, xanchor='center'),
)
fig.update_yaxes(range=[0, 120])
fig.show()

## まとめ・考察

EN ピーク（DTW 推定）から確定 LAG_PAIRS との DTW マッチングで JP 波及タイミングを推定した。

**手法選定の経緯**  
Prophet による JP 系列外挿も比較検討したが、JP データが上昇途中のキーワードに対しては  
ラグがほぼ 0〜マイナスとなり（"現在がピーク"と外挿してしまう）、全件で機能しないことを確認。  
EN 波形の形状類似度をもとにラグを推定する dtw_analog の方が理にかなっていると判断した。

**推定結果のまとめ**  
確定 LAG_PAIRS との比較では最近傍ラグは 11〜16ヶ月の範囲に収まっており、  
EN ピーク後 約 1 年 で JP 波及ピークを迎えると推定される。

In [ ]:
import os
os.makedirs('../docs', exist_ok=True)

# EN/JP 系列チャート
fig.write_html('../jp_timing.html')
fig.write_image('../docs/jp_timing.png', width=1200, height=340 * rows, scale=2)
print('保存: jp_timing.html / docs/jp_timing.png')

# サマリーテーブル
summary_cols = ['concept', 'EN keyword', 'JP keyword', 'EN peak (DTW)', 'JP peak (est)', 'lag (months)', 'nearest_pair']
pd.DataFrame(jp_timing_results)[summary_cols]